# PC 冒烟测试：CalmDataset d=10/20/50

使用 `calm_dataset.CalmDataset` 生成 `degree=1` 的线性高斯 SEM 数据，并通过 `fges_compat.TetradSearch.run_pc()` 调用 TETRAD PC + Fisher Z。这个 notebook 只验证 PC 导入、数据生成、运行和 endpoint matrix 转换是否正常。

In [2]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "calm_dataset.py").exists() and (path / "fges_compat.py").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from calm_dataset import CalmDataset, weight_to_binary_adj
from fges_compat import TetradSearch

print(f"REPO_ROOT: {REPO_ROOT}")
print("PC import: OK")

REPO_ROOT: C:\Users\super\DAG
PC import: OK


## 配置

`n_samples` 设为 2000，足够做 smoke test，又不会让 `d=50` 运行太久。需要更稳定结果时再增大样本量或 trial 数。

In [3]:
CFG = {
    "d_list": [10, 20, 50],
    "degree": 1.0,
    "n_samples": 10000,
    "graph_type": "ER",
    "sem_type": "gauss",
    "noise_ratio": 1.0,
    "noise_scale_mode": "variance",
    "seed": 20260509,
    "pc_alpha": 0.01,
    "pc_stable": True,
    "pc_depth": -1,
    "standardize": True,
}

CFG

{'d_list': [10, 20, 50],
 'degree': 1.0,
 'n_samples': 10000,
 'graph_type': 'ER',
 'sem_type': 'gauss',
 'noise_ratio': 1.0,
 'noise_scale_mode': 'variance',
 'seed': 20260509,
 'pc_alpha': 0.01,
 'pc_stable': True,
 'pc_depth': -1,
 'standardize': True}

## 工具函数

In [4]:
def standardize_columns(X: np.ndarray) -> np.ndarray:
    scale = X.std(axis=0, keepdims=True)
    scale = np.where(scale > 0, scale, 1.0)
    return (X - X.mean(axis=0, keepdims=True)) / scale


def tetrad_matrix_to_adj(df_result) -> np.ndarray:
    """TETRAD endpoint matrix -> 0/1 adjacency (ARROW=2, TAIL=3)."""
    mat = np.asarray(df_result.values if hasattr(df_result, "values") else df_result)
    d = mat.shape[0]
    G = np.zeros((d, d), dtype=int)
    for i in range(d):
        for j in range(i + 1, d):
            a, b = int(mat[i, j]), int(mat[j, i])
            if a == 2 and b == 3:
                G[i, j] = 1
            elif a == 3 and b == 2:
                G[j, i] = 1
            elif a == 3 and b == 3:
                G[i, j] = G[j, i] = 1
            elif a != 0 or b != 0:
                G[i, j] = G[j, i] = 1
    np.fill_diagonal(G, 0)
    return G


def skeleton_adj(G: np.ndarray) -> np.ndarray:
    return ((G + G.T) > 0).astype(int)


def skeleton_metrics(G_true: np.ndarray, G_est: np.ndarray) -> dict:
    true_skel = np.triu(skeleton_adj(G_true), k=1).astype(bool)
    est_skel = np.triu(skeleton_adj(G_est), k=1).astype(bool)
    tp = int(np.sum(true_skel & est_skel))
    fp = int(np.sum((~true_skel) & est_skel))
    fn = int(np.sum(true_skel & (~est_skel)))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    shd_skeleton = fp + fn
    return {
        "skeleton_precision": precision,
        "skeleton_recall": recall,
        "skeleton_shd": shd_skeleton,
    }


def run_pc_once(d: int, seed: int) -> dict:
    dataset = CalmDataset(
        n=CFG["n_samples"],
        d=d,
        graph_type=CFG["graph_type"],
        degree=CFG["degree"],
        sem_type=CFG["sem_type"],
        noise_ratio=CFG["noise_ratio"],
        noise_scale_mode=CFG["noise_scale_mode"],
        seed=seed,
    )
    X = standardize_columns(dataset.X) if CFG["standardize"] else dataset.X
    df = pd.DataFrame(X, columns=[f"x{i}" for i in range(d)]).astype("float64")

    t0 = time.perf_counter()
    search = TetradSearch(df)
    search.set_verbose(False)
    search.run_pc(alpha=CFG["pc_alpha"], stable=CFG["pc_stable"], depth=CFG["pc_depth"])
    endpoint_matrix = search.get_graph_to_matrix()
    runtime_sec = time.perf_counter() - t0

    G_true = weight_to_binary_adj(dataset.B)
    G_est = tetrad_matrix_to_adj(endpoint_matrix)
    row = {
        "d": d,
        "n_samples": CFG["n_samples"],
        "degree": CFG["degree"],
        "seed": seed,
        "runtime_sec": runtime_sec,
        "true_edges": int(G_true.sum()),
        "estimated_directed_entries": int(G_est.sum()),
        "estimated_skeleton_edges": int(np.triu(skeleton_adj(G_est), k=1).sum()),
        "endpoint_nonzero_pairs": int((endpoint_matrix.values != 0).sum() // 2),
    }
    row.update(skeleton_metrics(G_true, G_est))
    return row

## 运行 PC 冒烟测试

In [5]:
rows = []
for offset, d in enumerate(CFG["d_list"]):
    seed = CFG["seed"] + offset
    try:
        row = run_pc_once(d=d, seed=seed)
        row["status"] = "ok"
        row["message"] = ""
    except Exception as exc:
        row = {
            "d": d,
            "n_samples": CFG["n_samples"],
            "degree": CFG["degree"],
            "seed": seed,
            "runtime_sec": np.nan,
            "status": "failed",
            "message": f"{type(exc).__name__}: {exc}",
        }
    rows.append(row)
    print(f"[PC] d={d} status={row['status']} runtime={row['runtime_sec']:.3f}s message={row['message']}")

df_pc_smoke = pd.DataFrame(rows)
display(df_pc_smoke)

[PC] d=10 status=ok runtime=2.542s message=
[PC] d=20 status=ok runtime=0.751s message=
[PC] d=50 status=ok runtime=1.714s message=


,d,n_samples,degree,seed,runtime_sec,true_edges,estimated_directed_entries,estimated_skeleton_edges,endpoint_nonzero_pairs,skeleton_precision,skeleton_recall,skeleton_shd,status,message
0,10,10000,1.0,20260509,2.541729,10,14,9,9,1.00,0.90,1,ok,
1,20,10000,1.0,20260510,0.751047,20,33,20,20,1.00,1.00,0,ok,
2,50,10000,1.0,20260511,1.713742,50,58,50,50,0.86,0.86,14,ok,


## 判定

只要三行 `status` 都是 `ok`，说明 PC 算法路径、TETRAD JVM、Fisher Z、CalmDataset 数据生成和 endpoint 矩阵转换都能正常工作。